<a href="https://colab.research.google.com/github/ajanekukler-droid/e2cnn/blob/master/examples/model_try_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# General E(2)-Equivariant Steerable CNNs  -  A concrete example


In [2]:
import torch
%pip install e2cnn
from e2cnn import gspaces
from e2cnn import nn

from torchvision.transforms import InterpolationMode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.3/225.3 kB 7.9 MB/s eta 0:00:00


Finally, we build a **Steerable CNN** and try it MNIST.

Let's also use a group a bit larger: we now build a model equivariant to $8$ rotations.
We indicate the group of $N$ discrete rotations as $C_N$, i.e. the **cyclic group** of order $N$.
In this case, we will use $C_8$.

Because the inputs are still gray-scale images, the input type of the model is again a *scalar field*.

However, internally we use *regular fields*: this is equivalent to a *group-equivariant convolutional neural network*.

Finally, we build *invariant* features for the final classification task by pooling over the group using *Group Pooling*.

The final classification is performed by a two fully connected layers.

# The model

Here is the definition of our model:

In [3]:
class C8SteerableCNN(torch.nn.Module):

    def __init__(self, n_classes=10):

        super(C8SteerableCNN, self).__init__()

        # the model is equivariant under rotations by 45 degrees, modelled by C8
        self.r2_act = gspaces.Rot2dOnR2(N=8)

        # the input image is a scalar field, corresponding to the trivial representation
        in_type = nn.FieldType(self.r2_act, [self.r2_act.trivial_repr])

        # we store the input type for wrapping the images into a geometric tensor during the forward pass
        self.input_type = in_type

        # convolution 1
        # first specify the output type of the convolutional layer
        # we choose 24 feature fields, each transforming under the regular representation of C8
        out_type = nn.FieldType(self.r2_act, 24*[self.r2_act.regular_repr])
        self.block1 = nn.SequentialModule(
            nn.MaskModule(in_type, 29, margin=1),
            nn.R2Conv(in_type, out_type, kernel_size=7, padding=1, bias=False),
            nn.InnerBatchNorm(out_type),
            nn.ReLU(out_type, inplace=True)
        )

        # convolution 2
        # the old output type is the input type to the next layer
        in_type = self.block1.out_type
        # the output type of the second convolution layer are 48 regular feature fields of C8
        out_type = nn.FieldType(self.r2_act, 48*[self.r2_act.regular_repr])
        self.block2 = nn.SequentialModule(
            nn.R2Conv(in_type, out_type, kernel_size=5, padding=2, bias=False),
            nn.InnerBatchNorm(out_type),
            nn.ReLU(out_type, inplace=True)
        )
        self.pool1 = nn.SequentialModule(
            nn.PointwiseAvgPoolAntialiased(out_type, sigma=0.66, stride=2)
        )

        # convolution 3
        # the old output type is the input type to the next layer
        in_type = self.block2.out_type
        # the output type of the third convolution layer are 48 regular feature fields of C8
        out_type = nn.FieldType(self.r2_act, 48*[self.r2_act.regular_repr])
        self.block3 = nn.SequentialModule(
            nn.R2Conv(in_type, out_type, kernel_size=5, padding=2, bias=False),
            nn.InnerBatchNorm(out_type),
            nn.ReLU(out_type, inplace=True)
        )

        # convolution 4
        # the old output type is the input type to the next layer
        in_type = self.block3.out_type
        # the output type of the fourth convolution layer are 96 regular feature fields of C8
        out_type = nn.FieldType(self.r2_act, 96*[self.r2_act.regular_repr])
        self.block4 = nn.SequentialModule(
            nn.R2Conv(in_type, out_type, kernel_size=5, padding=2, bias=False),
            nn.InnerBatchNorm(out_type),
            nn.ReLU(out_type, inplace=True)
        )
        self.pool2 = nn.SequentialModule(
            nn.PointwiseAvgPoolAntialiased(out_type, sigma=0.66, stride=2)
        )

        # convolution 5
        # the old output type is the input type to the next layer
        in_type = self.block4.out_type
        # the output type of the fifth convolution layer are 96 regular feature fields of C8
        out_type = nn.FieldType(self.r2_act, 96*[self.r2_act.regular_repr])
        self.block5 = nn.SequentialModule(
            nn.R2Conv(in_type, out_type, kernel_size=5, padding=2, bias=False),
            nn.InnerBatchNorm(out_type),
            nn.ReLU(out_type, inplace=True)
        )

        # convolution 6
        # the old output type is the input type to the next layer
        in_type = self.block5.out_type
        # the output type of the sixth convolution layer are 64 regular feature fields of C8
        out_type = nn.FieldType(self.r2_act, 64*[self.r2_act.regular_repr])
        self.block6 = nn.SequentialModule(
            nn.R2Conv(in_type, out_type, kernel_size=5, padding=1, bias=False),
            nn.InnerBatchNorm(out_type),
            nn.ReLU(out_type, inplace=True)
        )
        self.pool3 = nn.PointwiseAvgPoolAntialiased(out_type, sigma=0.66, stride=1, padding=0)

        self.gpool = nn.GroupPooling(out_type)

        # number of output channels
        c = self.gpool.out_type.size

        # Fully Connected
        self.fully_net = torch.nn.Sequential(
            torch.nn.Linear(c, 64),
            torch.nn.BatchNorm1d(64),
            torch.nn.ELU(inplace=True),
            torch.nn.Linear(64, n_classes),
        )

    def forward(self, input: torch.Tensor):
        # wrap the input tensor in a GeometricTensor
        # (associate it with the input type)
        x = nn.GeometricTensor(input, self.input_type)

        # apply each equivariant block

        # Each layer has an input and an output type
        # A layer takes a GeometricTensor in input.
        # This tensor needs to be associated with the same representation of the layer's input type
        #
        # The Layer outputs a new GeometricTensor, associated with the layer's output type.
        # As a result, consecutive layers need to have matching input/output types
        x = self.block1(x)
        x = self.block2(x)
        x = self.pool1(x)

        x = self.block3(x)
        x = self.block4(x)
        x = self.pool2(x)

        x = self.block5(x)
        x = self.block6(x)

        # pool over the spatial dimensions
        x = self.pool3(x)

        # pool over the group
        x = self.gpool(x)

        # unwrap the output GeometricTensor
        # (take the Pytorch tensor and discard the associated representation)
        x = x.tensor

        # classify with the final fully connected layers)
        x = self.fully_net(x.reshape(x.shape[0], -1))

        return x

Let's try the model on *rotated* MNIST

In [4]:
# download the dataset
!wget -nc http://www.iro.umontreal.ca/~lisa/icml2007data/mnist_rotation_new.zip
# uncompress the zip file
!unzip -n mnist_rotation_new.zip -d mnist_rotation_new

--2026-05-17 11:36:49--  http://www.iro.umontreal.ca/~lisa/icml2007data/mnist_rotation_new.zip
Resolving www.iro.umontreal.ca (www.iro.umontreal.ca)... 132.204.26.36
Connecting to www.iro.umontreal.ca (www.iro.umontreal.ca)|132.204.26.36|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://www.iro.umontreal.ca/~lisa/icml2007data/mnist_rotation_new.zip [following]
--2026-05-17 11:36:50--  https://www.iro.umontreal.ca/~lisa/icml2007data/mnist_rotation_new.zip
Connecting to www.iro.umontreal.ca (www.iro.umontreal.ca)|132.204.26.36|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 58424278 (56M) [application/zip]
Saving to: ‘mnist_rotation_new.zip’

mnist_rotation_new. 100%[===================>]  55.72M  33.3MB/s    in 1.7s    

2026-05-17 11:36:52 (33.3 MB/s) - ‘mnist_rotation_new.zip’ saved [58424278/58424278]

Archive:  mnist_rotation_new.zip
  inflating: mnist_rotation_new/mnist_all_rotation_normalized_float_train_val

In [5]:
from torch.utils.data import Dataset
from torchvision.transforms import RandomRotation
from torchvision.transforms import Pad
from torchvision.transforms import Resize
from torchvision.transforms import ToTensor
from torchvision.transforms import Compose

import numpy as np

from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'


In [6]:
device

'cuda'

Build the dataset

In [7]:
class MnistRotDataset(Dataset):

    def __init__(self, mode, transform=None):
        assert mode in ['train', 'test']

        if mode == "train":
            file = "mnist_rotation_new/mnist_all_rotation_normalized_float_train_valid.amat"
        else:
            file = "mnist_rotation_new/mnist_all_rotation_normalized_float_test.amat"

        self.transform = transform

        data = np.loadtxt(file, delimiter=' ')

        self.images = data[:, :-1].reshape(-1, 28, 28).astype(np.float32)
        self.labels = data[:, -1].astype(np.int64)
        self.num_samples = len(self.labels)

    def __getitem__(self, index):
        image, label = self.images[index], self.labels[index]
        image = Image.fromarray(image)
        if self.transform is not None:
            image = self.transform(image)
        return image, label

    def __len__(self):
        return len(self.labels)

# images are padded to have shape 29x29.
# this allows to use odd-size filters with stride 2 when downsampling a feature map in the model
pad = Pad((0, 0, 1, 1), fill=0)

# to reduce interpolation artifacts (e.g. when testing the model on rotated images),
# we upsample an image by a factor of 3, rotate it and finally downsample it again
resize1 = Resize(87)
resize2 = Resize(29)

totensor = ToTensor()

Let's build the model

In [8]:
model = C8SteerableCNN().to(device)

/usr/local/lib/python3.12/dist-packages/e2cnn/nn/modules/r2_conv/basisexpansion_singleblock.py:80: UserWarning: indexing with dtype torch.uint8 is now deprecated, please use a dtype torch.bool instead. (Triggered internally at /pytorch/aten/src/ATen/native/IndexingUtils.h:37.)
  full_mask[mask] = norms.to(torch.uint8)


The model is now randomly initialized.
Therefore, we do not expect it to produce the right class probabilities.

However, the model should still produce the same output for rotated versions of the same image.
This is true for rotations by multiples of $\frac{\pi}{2}$, but is only approximate for rotations by $\frac{\pi}{4}$.

Let's test it on a random test image:
we feed eight rotated versions of the first image in the test set and print the output logits of the model for each of them.

In [38]:
import random

def inference_random_angle(model: torch.nn.Module, x: Image, plotrange):
    # evaluate the `model` on 8 rotated versions of the input image `x`
    model.eval()

    wrmup = model(torch.randn(1, 1, 29, 29).to(device))
    del wrmup

    x = resize1(pad(x))

    random_angle = random.randint(1,360)
    with torch.no_grad():
        x_transformed = totensor(resize2(x.rotate(random_angle, Image.BILINEAR))).reshape(1, 1, 29, 29)
        x_transformed = x_transformed.to(device)

        y = model(x_transformed)
        y = list(y.to('cpu').numpy().squeeze())
        max_value = max(y)
        pred_label = y.index(max_value)
        return pred_label


def test_model(model: torch.nn.Module, x: Image):
    # evaluate the `model` on 8 rotated versions of the input image `x`
    model.eval()

    wrmup = model(torch.randn(1, 1, 29, 29).to(device))
    del wrmup

    x = resize1(pad(x))

    print()
    print('##########################################################################################')
    header = 'angle |  ' + '  '.join(["{:6d}".format(d) for d in range(10)])
    print(header)
    with torch.no_grad():
        for r in range(8):
            x_transformed = totensor(resize2(x.rotate(r*45., Image.BILINEAR))).reshape(1, 1, 29, 29)
            x_transformed = x_transformed.to(device)

            y = model(x_transformed)
            y = y.to('cpu').numpy().squeeze()

            angle = r * 45
            print("{:5d} : {}".format(angle, y))
    print('##########################################################################################')
    print()



In [39]:
example_inference = raw_mnist_test[0]
label_pred = inference_random_angle(model, example_inference[0])

print(example_inference[1], label_pred)

6 6


In [10]:
# build the test set
raw_mnist_test = MnistRotDataset(mode='test')

In [11]:
# retrieve the first image from the test set
x, y = next(iter(raw_mnist_test))

# evaluate the model
test_model(model, x)


##########################################################################################
angle |       0       1       2       3       4       5       6       7       8       9
    0 : [ 0.2716  0.2191 -0.1859 -0.2281  0.4548 -0.2909 -0.5944  0.0886 -0.0852 -0.4045]
   45 : [ 0.2601  0.2264 -0.1849 -0.2318  0.4366 -0.2883 -0.5865  0.0969 -0.1088 -0.3995]
   90 : [ 0.2716  0.2191 -0.1859 -0.2281  0.4548 -0.2909 -0.5944  0.0886 -0.0852 -0.4045]
  135 : [ 0.2601  0.2264 -0.1849 -0.2318  0.4366 -0.2883 -0.5865  0.0969 -0.1088 -0.3995]
  180 : [ 0.2716  0.2191 -0.1859 -0.2281  0.4548 -0.2909 -0.5944  0.0886 -0.0852 -0.4045]
  225 : [ 0.2601  0.2264 -0.1849 -0.2318  0.4366 -0.2883 -0.5865  0.0969 -0.1088 -0.3995]
  270 : [ 0.2716  0.2191 -0.1859 -0.2281  0.4548 -0.2909 -0.5944  0.0886 -0.0852 -0.4045]
  315 : [ 0.2601  0.2264 -0.1849 -0.2318  0.4366 -0.2883 -0.5865  0.0969 -0.1088 -0.3995]
##########################################################################################



The output of the model is already almost invariant.
However, we still observe small fluctuations in the outputs.

This is because the model contains some operations which might break equivariance.
For instance, every convolution includes a padding of $2$ pixels per side. This is adds information about the actual orientation of the grid where the image/feature map is sampled because the padding is not rotated with the image.

During training, the model will observe rotated patterns and will learn to ignore the noise coming from the padding.

So, let's train the model now.
The model is exactly the same used to train a normal *PyTorch* architecture:

In [12]:
train_transform = Compose([
    pad,
    resize1,
    RandomRotation(180, interpolation=InterpolationMode.BILINEAR, expand=False),
    resize2,
    totensor,
])

mnist_train = MnistRotDataset(mode='train', transform=train_transform)
train_loader = torch.utils.data.DataLoader(mnist_train, batch_size=64)


test_transform = Compose([
    pad,
    totensor,
])
mnist_test = MnistRotDataset(mode='test', transform=test_transform)
test_loader = torch.utils.data.DataLoader(mnist_test, batch_size=64)

loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=1e-5)

In [14]:
import torch

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
device

device(type='cuda', index=0)

In [15]:
print('Cuda available:', torch.cuda.is_available())
print('Cuda device count:', torch.cuda.device_count())
print('Current device:', torch.cuda.current_device())
print('Device 0:', torch.cuda.device(0))
print('Device name:', torch.cuda.get_device_name(0))

Cuda available: True
Cuda device count: 1
Current device: 0
Device 0: <torch.cuda.device object at 0x7abda71fbf20>
Device name: Tesla T4


In [16]:
train_loader

In [17]:
from tqdm import tqdm

for epoch in tqdm(range(31)):
    model.train()
    for i, (x, t) in enumerate(train_loader):

        optimizer.zero_grad()

        x = x.to(device)
        t = t.to(device)

        y = model(x)

        loss = loss_function(y, t)

        loss.backward()

        optimizer.step()

    if epoch % 10 == 0:
        total = 0
        correct = 0
        with torch.no_grad():
            model.eval()
            for i, (x, t) in enumerate(test_loader):

                x = x.to(device)
                t = t.to(device)

                y = model(x)

                _, prediction = torch.max(y.data, 1)
                total += t.shape[0]
                correct += (prediction == t).sum().item()
        print(f"epoch {epoch} | test accuracy: {correct/total*100.}")


  3%|▎         | 1/31 [03:05<1:32:40, 185.33s/it]

epoch 0 | test accuracy: 91.976


 35%|███▌      | 11/31 [18:25<38:28, 115.41s/it]

epoch 10 | test accuracy: 97.74000000000001


 68%|██████▊   | 21/31 [33:45<19:08, 114.83s/it]

epoch 20 | test accuracy: 97.924


100%|██████████| 31/31 [49:06<00:00, 95.06s/it] 

epoch 30 | test accuracy: 97.88799999999999


In [25]:
import os

if not os.path.exists("models"):
  os.mkdir("models")
torch.save(model.state_dict(), "/content/models/1st-statedict.pth")

In [36]:
example_inference = raw_mnist_test[0]
label_pred = inference_random_angle(model, example_inference[0])

print(example_inference[1], label_pred)

6 [np.float32(-1.2161374), np.float32(-0.21643403), np.float32(-0.9964822), np.float32(-0.96985006), np.float32(0.6890448), np.float32(-0.9520353), np.float32(9.6495075), np.float32(-0.12294701), np.float32(-2.2681675), np.float32(-0.6079827)]


In [18]:
# retrieve the first image from the test set
x, y = next(iter(raw_mnist_test))


# evaluate the model
test_model(model, x)


##########################################################################################
angle |       0       1       2       3       4       5       6       7       8       9
    0 : [-1.2275 -0.2711 -0.7684 -0.9039  0.7133 -1.0873  9.502  -0.0802 -2.1812 -0.6263]
   45 : [-1.3071 -0.1918 -0.6772 -0.9601  0.5609 -1.188   9.7173 -0.6114 -2.1237 -0.1645]
   90 : [-1.2275 -0.2711 -0.7684 -0.9039  0.7133 -1.0873  9.502  -0.0802 -2.1812 -0.6263]
  135 : [-1.3071 -0.1918 -0.6772 -0.9601  0.5609 -1.188   9.7173 -0.6114 -2.1237 -0.1645]
  180 : [-1.2275 -0.2711 -0.7684 -0.9039  0.7133 -1.0873  9.502  -0.0802 -2.1812 -0.6263]
  225 : [-1.3071 -0.1918 -0.6772 -0.9601  0.5609 -1.188   9.7173 -0.6114 -2.1237 -0.1645]
  270 : [-1.2275 -0.2711 -0.7684 -0.9039  0.7133 -1.0873  9.502  -0.0802 -2.1812 -0.6263]
  315 : [-1.3071 -0.1918 -0.6772 -0.9601  0.5609 -1.188   9.7173 -0.6114 -2.1237 -0.1645]
##########################################################################################



In [40]:
# ── 1. Install Kaggle API ──────────────────────────────────────────────────
!pip install kaggle -q

# ── 2. Upload kaggle.json directly in Colab (no Drive needed) ─────────────
# A file picker will appear — select your kaggle.json from your computer
import os
from google.colab import files

os.makedirs('/root/.kaggle', exist_ok=True)
files.upload()  # ← pick your kaggle.json here

os.system('mv kaggle.json /root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# ── 3. Download the dataset ────────────────────────────────────────────────
!kaggle datasets download -d saiteja0101/rotated-mnist -p /content/rotated-mnist --unzip

# ── 4. Load the .amat files ────────────────────────────────────────────────
import numpy as np

def load_amat(filepath):
    data = np.loadtxt(filepath)
    X = data[:, :-1]
    y = data[:, -1]
    return X, y

print("Loading training set... (this may take a minute)")
X_train, y_train = load_amat('/content/rotated-mnist/mnist_all_rotation_normalized_float_train_valid.amat')

print("Loading test set...")
X_test, y_test = load_amat('/content/rotated-mnist/mnist_all_rotation_normalized_float_test.amat')

# ── 5. Sanity check ────────────────────────────────────────────────────────
print(f"\nX_train : {X_train.shape}  |  y_train : {y_train.shape}")
print(f"X_test  : {X_test.shape}   |  y_test  : {y_test.shape}")
print(f"Pixel range : [{X_train.min():.3f}, {X_train.max():.3f}]")
print(f"Classes     : {np.unique(y_train).astype(int)}")

# ── 6. Peek at samples ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(X_train[i].reshape(28, 28), cmap='gray')
    ax.set_title(f"Label: {int(y_train[i])}")
    ax.axis('off')
plt.suptitle("Rotated MNIST samples")
plt.tight_layout()
plt.show()

KeyboardInterrupt: 